# 03 — Statistical Tests

Friedman test, Nemenyi post-hoc, pairwise Wilcoxon, and critical difference diagrams.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.statistical_tests import (
    friedman_test, nemenyi_test, pairwise_wilcoxon,
    compute_average_ranks, plot_cd_diagram,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

test_df = pd.read_csv('../results/aggregated/test_results.csv')

In [ ]:
# Separate by task type and run tests
tests_config = [
    ('binary', 'roc_auc', True),
    ('multiclass', 'accuracy', True),
    ('regression', 'rmse', False),
]

for task_type, metric, higher_is_better in tests_config:
    subset = test_df[test_df['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        print(f"Skipping {task_type}: no data")
        continue

    pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)

    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        print(f"Skipping {task_type}: not enough data (need >=3 datasets and >=3 models)")
        continue

    print(f"\n{'='*60}")
    print(f"{task_type.upper()} — {metric}")
    print(f"{'='*60}")

    # Friedman test
    friedman = friedman_test(pivot)
    print(f"\nFriedman test: chi2={friedman['statistic']:.3f}, p={friedman['p_value']:.6f}")
    print(f"Reject null (alpha=0.05): {friedman['reject_null']}")

    # Average ranks
    ranks = compute_average_ranks(pivot, higher_is_better=higher_is_better)
    print(f"\nAverage Ranks:")
    for model, rank in ranks.items():
        print(f"  {model}: {rank:.2f}")

In [ ]:
# Critical Difference Diagrams
for task_type, metric, higher_is_better in tests_config:
    subset = test_df[test_df['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        continue

    pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        continue

    fig = plot_cd_diagram(
        pivot,
        title=f'Critical Difference — {task_type.title()} ({metric})',
        higher_is_better=higher_is_better,
        save_path=f'../results/figures/cd_diagram_{task_type}.png',
    )
    plt.show()

In [ ]:
# Pairwise Wilcoxon signed-rank tests
for task_type, metric, higher_is_better in tests_config:
    subset = test_df[test_df['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        continue

    pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        continue

    p_values = pairwise_wilcoxon(pivot)

    print(f"\n=== Wilcoxon p-values: {task_type} ({metric}) ===")

    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(p_values, dtype=bool))
    sns.heatmap(p_values, annot=True, fmt='.4f', cmap='RdYlGn_r',
                mask=mask, vmin=0, vmax=0.1, linewidths=0.5)
    plt.title(f'Pairwise Wilcoxon p-values — {task_type.title()}')
    plt.tight_layout()
    plt.savefig(f'../results/figures/wilcoxon_{task_type}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Nemenyi post-hoc test
for task_type, metric, higher_is_better in tests_config:
    subset = test_df[test_df['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        continue

    pivot = subset.pivot(index='dataset', columns='model', values=metric).dropna(axis=1)
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        continue

    nemenyi = nemenyi_test(pivot, higher_is_better=higher_is_better)
    print(f"\n=== Nemenyi post-hoc: {task_type} ===")
    display(nemenyi)